# Age and Gender Prediction Model Training
This notebook trains deep learning models for age and gender prediction using the UTKFace dataset.

**Requirements:**
- Google Colab with GPU enabled (Runtime → Change runtime type → T4 GPU)
- Dataset will be automatically downloaded from Kaggle using kagglehub

## Step 1: Install Packages

In [ ]:
!pip install kagglehub tqdm scikit-learn

## Step 2: Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import kagglehub
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt

## Step 3: Dataset Loading Function

In [ ]:
def load_utk_dataset(dataset_path, max_images=12000):
    """Load UTKFace dataset with memory optimization"""
    ages, genders, images = [], [], []
    print("Loading images from:", dataset_path)
    
    file_list = [f for f in os.listdir(dataset_path) if f.endswith('.jpg') or f.endswith('.chip.jpg')]
    # Limit number of images to avoid memory issues in Colab
    file_list = file_list[:max_images]
    print(f"Loading {len(file_list)} images (max_images={max_images})")
    
    for filename in tqdm(file_list):
        try:
            parts = filename.split('_')
            if len(parts) >= 2:
                age, gender = int(parts[0]), int(parts[1])
                img = cv2.imread(os.path.join(dataset_path, filename))
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (128, 128))
                    img = img.astype(np.float32) / 255.0  # Use float32 to reduce memory
                    images.append(img)
                    ages.append(age)
                    genders.append(gender)
        except:
            continue
    
    return np.array(images, dtype=np.float32), np.array(ages), np.array(genders)

## Step 4: Gender Model

In [ ]:
def create_gender_model():
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Conv2D(256, (3, 3), activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(512, activation='relu'), layers.Dropout(0.5),
        layers.Dense(256, activation='relu'), layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

## Step 5: Age Model

In [ ]:
def create_age_model():
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Conv2D(256, (3, 3), activation='relu'),
        layers.BatchNormalization(), layers.MaxPooling2D((2, 2)), layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(512, activation='relu'), layers.Dropout(0.5),
        layers.Dense(256, activation='relu'), layers.Dropout(0.5),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

## Step 6: Download Dataset

In [ ]:
print("Downloading UTKFace dataset from Kaggle...")
path = kagglehub.dataset_download("jangedoo/utkface-new")
print(f"Dataset downloaded to: {path}")

# Check if images are in a UTKFace subdirectory
utkface_path = os.path.join(path, 'UTKFace')
if os.path.exists(utkface_path):
    path = utkface_path
    print(f"Using images from: {path}")
else:
    print(f"Using base path: {path}")

## Step 7: Load Dataset

In [ ]:
images, ages, genders = load_utk_dataset(path, max_images=12000)
print(f"\nLoaded {len(images)} images")
if len(images) > 0:
    print(f"Memory used: {images.nbytes / (1024**3):.2f} GB")
    print(f"Age range: {ages.min()}-{ages.max()} years")
    print(f"Male: {np.sum(genders==0)}, Female: {np.sum(genders==1)}")
else:
    print("⚠️ No images loaded! Check the dataset path.")

## Step 8: Split Data

In [ ]:
X_train, X_test, y_age_train, y_age_test, y_gender_train, y_gender_test = train_test_split(
    images, ages, genders, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Step 9: Train Gender Model

In [ ]:
import gc
print("\n" + "="*60)
print("TRAINING GENDER MODEL")
print("="*60)

# Clear memory before training
gc.collect()

gender_model = create_gender_model()
gender_model.summary()
gender_history = gender_model.fit(
    X_train, y_gender_train,
    validation_data=(X_test, y_gender_test),
    epochs=30,  # Reduced from 50
    batch_size=16,  # Reduced from 32 to save memory
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint('gender_classifier_model.keras', save_best_only=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ],
    verbose=1
)
print("✓ Gender model complete!")

## Step 10: Train Age Model

In [ ]:
print("\n" + "="*60)
print("TRAINING AGE MODEL")
print("="*60)

# Clear memory before training
del gender_model
gc.collect()

age_model = create_age_model()
age_model.summary()
age_history = age_model.fit(
    X_train, y_age_train,
    validation_data=(X_test, y_age_test),
    epochs=30,  # Reduced from 50
    batch_size=16,  # Reduced from 32 to save memory
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint('age_prediction_model.keras', save_best_only=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ],
    verbose=1
)
print("✓ Age model complete!")

## Step 11: Plot Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0, 0].plot(gender_history.history['accuracy'], label='Train')
axes[0, 0].plot(gender_history.history['val_accuracy'], label='Val')
axes[0, 0].set_title('Gender Accuracy')
axes[0, 0].legend()
axes[0, 1].plot(gender_history.history['loss'], label='Train')
axes[0, 1].plot(gender_history.history['val_loss'], label='Val')
axes[0, 1].set_title('Gender Loss')
axes[0, 1].legend()
axes[1, 0].plot(age_history.history['mae'], label='Train')
axes[1, 0].plot(age_history.history['val_mae'], label='Val')
axes[1, 0].set_title('Age MAE')
axes[1, 0].legend()
axes[1, 1].plot(age_history.history['loss'], label='Train')
axes[1, 1].plot(age_history.history['val_loss'], label='Val')
axes[1, 1].set_title('Age Loss')
axes[1, 1].legend()
plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

## Step 12: Final Results

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Gender Accuracy: {max(gender_history.history['val_accuracy']):.4f}")
print(f"Age MAE: {min(age_history.history['val_mae']):.2f} years")
print("\nModels saved: gender_classifier_model.keras, age_prediction_model.keras")